In [ ]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.callbacks import LearningRateScheduler
import numpy as np
import matplotlib.pyplot as plt

batch_size = 128
num_classes = 10
epochs = 20

# Learning rate scheduler
def lr_scheduler(epoch):
    return 0.1 * np.exp(-epoch / 10.)

# Load the data
def load(f):
    return np.load(f)['arr_0']

x_train = load('X_kannada_MNIST_train.npz')
x_test = load('X_kannada_MNIST_test.npz')
y_train = load('y_kannada_MNIST_train.npz')
y_test = load('y_kannada_MNIST_test.npz')

input_shape = x_train.shape[1:] + (1,)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('{} train samples, {} test samples'.format(len(x_train), len(x_test)))

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# Function to display images
def display_images(images, labels, num_images=5, predictions=None):
    plt.figure(figsize=(15, 5))
    for i in range(num_images):
        plt.subplot(2, num_images, i+1)
        plt.imshow(images[i].reshape(28, 28), cmap='gray')
        plt.title('Original: {}'.format(np.argmax(labels[i])))
        plt.axis('off')

        if predictions is not None:
            plt.subplot(2, num_images, i+num_images+1)
            plt.text(0.5, 0.5, str(predictions[i]), fontsize=20, ha='center')  # Display predicted label
            plt.title('Predicted: {}'.format(predictions[i]))
            plt.axis('off')
    plt.show()

# Display sample images
display_images(x_test, y_test)

# Process predictions to generate images
def process_predictions(predictions):
    predicted_labels = []
    for prediction in predictions:
        # Reshape the one-hot encoded label into a 1D array of size 10
        predicted_label = np.argmax(prediction)
        predicted_labels.append(predicted_label)
    return predicted_labels

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.55))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adadelta(),
              metrics=['accuracy'])

# Learning rate scheduler callback
lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

# Fit the model with learning rate scheduling, batch normalization, and dropout
history = model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test),
          callbacks=[lr_scheduler_callback])

train_score = model.evaluate(x_train, y_train, verbose=0)
test_score = model.evaluate(x_test, y_test, verbose=0)
print('Train loss:', train_score[0])
print('Train accuracy:', train_score[1])
print('Test loss:', test_score[0])
print('Test accuracy:', test_score[1])

# Predictions on a subset of test data
num_images_to_predict = 5
predicted_images = process_predictions(model.predict(x_test[:num_images_to_predict]))

# Display original and predicted images
display_images(x_test[:num_images_to_predict], y_test[:num_images_to_predict], predictions=predicted_images)

In [ ]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.callbacks import LearningRateScheduler
import numpy as np
import matplotlib.pyplot as plt

batch_size = 128
num_classes = 10
epochs = 30  # Increased epochs

# Learning rate scheduler
def lr_scheduler(epoch):
    return 0.01 * np.exp(-epoch / 10.)  # Adjusted learning rate

# Load the data
def load(f):
    return np.load(f)['arr_0']

x_train = load('X_kannada_MNIST_train.npz')
x_test = load('X_kannada_MNIST_test.npz')
y_train = load('y_kannada_MNIST_train.npz')
y_test = load('y_kannada_MNIST_test.npz')

input_shape = x_train.shape[1:] + (1,)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('{} train samples, {} test samples'.format(len(x_train), len(x_test)))

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(128, (3, 3), activation='relu'))  # Added another convolutional layer
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), activation='relu'))  # Added another convolutional layer
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))  # Increased number of units in dense layer
model.add(BatchNormalization())
model.add(Dropout(0.5))  # Increased dropout rate
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(),  # Changed optimizer to Adam
              metrics=['accuracy'])

# Learning rate scheduler callback
lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

# Fit the model with learning rate scheduling, batch normalization, and dropout
history = model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test),
          callbacks=[lr_scheduler_callback])

train_score = model.evaluate(x_train, y_train, verbose=0)
test_score = model.evaluate(x_test, y_test, verbose=0)
print('Train loss:', train_score[0])
print('Train accuracy:', train_score[1])
print('Test loss:', test_score[0])
print('Test accuracy:', test_score[1])

# Predictions on a subset of test data
num_images_to_predict = 5
predicted_images = process_predictions(model.predict(x_test[:num_images_to_predict]))

# Display original and predicted images
display_images(x_test[:num_images_to_predict], y_test[:num_images_to_predict], predictions=predicted_images)

In [ ]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.callbacks import LearningRateScheduler, EarlyStopping
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt

batch_size = 128
num_classes = 10
epochs = 100

# Learning rate scheduler
def lr_scheduler(epoch):
    return 0.01 * np.exp(-epoch / 10.)

# Load the data
def load(f):
    return np.load(f)['arr_0']

x_train = load('X_kannada_MNIST_train.npz')
x_test = load('X_kannada_MNIST_test.npz')
y_train = load('y_kannada_MNIST_train.npz')
y_test = load('y_kannada_MNIST_test.npz')

input_shape = x_train.shape[1:] + (1,)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('{} train samples, {} test samples'.format(len(x_train), len(x_test)))

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Define the model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(BatchNormalization())  # Batch normalization after each Conv2D
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))  # Added a Dense layer
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(),
              metrics=['accuracy'])

# Learning rate scheduler callback
lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

# Fit the model with data augmentation and callbacks
history = model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),
                    steps_per_epoch=len(x_train) / batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_test, y_test),
                    callbacks=[lr_scheduler_callback])

# Evaluate the model
train_score = model.evaluate(x_train, y_train, verbose=0)
test_score = model.evaluate(x_test, y_test, verbose=0)
print('Train loss:', train_score[0])
print('Train accuracy:', train_score[1])
print('Test loss:', test_score[0])
print('Test accuracy:', test_score[1])

60000 train samples, 10000 test samples
Epoch 1/100
468/468 [==============================] - 28s 44ms/step - loss: 1.0085 - accuracy: 0.6323 - val_loss: 1.8332 - val_accuracy: 0.5350 - lr: 0.0100
Epoch 2/100
468/468 [==============================] - 19s 41ms/step - loss: 0.6013 - accuracy: 0.7838 - val_loss: 0.4938 - val_accuracy: 0.8211 - lr: 0.0090
Epoch 3/100
468/468 [==============================] - 20s 42ms/step - loss: 0.4966 - accuracy: 0.8238 - val_loss: 0.4320 - val_accuracy: 0.8549 - lr: 0.0082
Epoch 4/100
468/468 [==============================] - 20s 42ms/step - loss: 0.4461 - accuracy: 0.8444 - val_loss: 0.4215 - val_accuracy: 0.8525 - lr: 0.0074
Epoch 5/100
468/468 [==============================] - 20s 43ms/step - loss: 0.4116 - accuracy: 0.8574 - val_loss: 0.4053 - val_accuracy: 0.8609 - lr: 0.0067
Epoch 6/100
468/468 [==============================] - 20s 43ms/step - loss: 0.3839 - accuracy: 0.8669 - val_loss: 0.4569 - val_accuracy: 0.8342 - lr: 0.0061
Epoch 7/100


KeyboardInterrupt: 

In [ ]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.callbacks import LearningRateScheduler, ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt

batch_size = 128
num_classes = 10
epochs = 100

# Learning rate scheduler
def lr_scheduler(epoch):
    return 0.01 * np.exp(-epoch / 10.)

# Load the data
def load(f):
    return np.load(f)['arr_0']

x_train = load('X_kannada_MNIST_train.npz')
x_test = load('X_kannada_MNIST_test.npz')
y_train = load('y_kannada_MNIST_train.npz')
y_test = load('y_kannada_MNIST_test.npz')

input_shape = x_train.shape[1:] + (1,)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('{} train samples, {} test samples'.format(len(x_train), len(x_test)))

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Define the model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(BatchNormalization())  # Batch normalization after each Conv2D
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(512, activation='relu'))  # Added a Dense layer
model.add(Dropout(0.45))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(),
              metrics=['accuracy'])

# Learning rate scheduler callback
# lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

# # Reduce learning rate when a metric has stopped improving
# reduce_lr_callback = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, min_lr=0.0001)

# Fit the model with data augmentation and callbacks
history = model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),
                    steps_per_epoch=len(x_train) / batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_test, y_test),
                    # callbacks=[lr_scheduler_callback, reduce_lr_callback]
)

# Evaluate the model
train_score = model.evaluate(x_train, y_train, verbose=0)
test_score = model.evaluate(x_test, y_test, verbose=0)
print('Train loss:', train_score[0])
print('Train accuracy:', train_score[1])
print('Test loss:', test_score[0])
print('Test accuracy:', test_score[1])


60000 train samples, 10000 test samples
Epoch 1/100
468/468 [==============================] - 22s 41ms/step - loss: 0.9545 - accuracy: 0.6464 - val_loss: 1.1018 - val_accuracy: 0.6471
Epoch 2/100
468/468 [==============================] - 20s 43ms/step - loss: 0.5261 - accuracy: 0.8094 - val_loss: 0.3903 - val_accuracy: 0.8604
Epoch 3/100
468/468 [==============================] - 19s 41ms/step - loss: 0.4343 - accuracy: 0.8455 - val_loss: 0.3981 - val_accuracy: 0.8530
Epoch 4/100
468/468 [==============================] - 19s 40ms/step - loss: 0.3827 - accuracy: 0.8649 - val_loss: 0.3345 - val_accuracy: 0.8767
Epoch 5/100
468/468 [==============================] - 20s 42ms/step - loss: 0.3551 - accuracy: 0.8750 - val_loss: 0.3401 - val_accuracy: 0.8839
Epoch 6/100
468/468 [==============================] - 19s 41ms/step - loss: 0.3332 - accuracy: 0.8826 - val_loss: 0.3003 - val_accuracy: 0.8931
Epoch 7/100
468/468 [==============================] - 19s 40ms/step - loss: 0.3163 - accu

In [ ]:
from __future__ import print_function
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.callbacks import LearningRateScheduler, EarlyStopping, ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt

batch_size = 128
num_classes = 10
epochs = 100

# Learning rate scheduler
def lr_scheduler(epoch):
    return 0.01 * np.exp(-epoch / 10.)

# Load the data
def load(f):
    return np.load(f)['arr_0']

x_train = load('X_kannada_MNIST_train.npz')
x_test = load('X_kannada_MNIST_test.npz')
y_train = load('y_kannada_MNIST_train.npz')
y_test = load('y_kannada_MNIST_test.npz')

input_shape = x_train.shape[1:] + (1,)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('{} train samples, {} test samples'.format(len(x_train), len(x_test)))

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    # horizontal_flip=True,
    # vertical_flip=True,
    fill_mode='nearest'
)

# Define the model
model = Sequential()
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(BatchNormalization())  # Batch normalization after each Conv2D
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Dropout(0.1))
model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Dropout(0.1))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(),
              metrics=['accuracy'])

# Learning rate scheduler callback
lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

# Reduce learning rate on plateau
reduce_lr_callback = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.0001)

# Fit the model with data augmentation and callbacks
history = model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),
                    steps_per_epoch=len(x_train) / batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_test, y_test),
                    callbacks=[lr_scheduler_callback, reduce_lr_callback])

# Evaluate the model
train_score = model.evaluate(x_train, y_train, verbose=0)
test_score = model.evaluate(x_test, y_test, verbose=0)
print('Train loss:', train_score[0])
print('Train accuracy:', train_score[1])
print('Test loss:', test_score[0])
print('Test accuracy:', test_score[1])


60000 train samples, 10000 test samples
Epoch 1/100
468/468 [==============================] - 23s 45ms/step - loss: 0.6621 - accuracy: 0.7764 - val_loss: 0.2694 - val_accuracy: 0.9089 - lr: 0.0100
Epoch 2/100
468/468 [==============================] - 21s 45ms/step - loss: 0.3335 - accuracy: 0.8828 - val_loss: 0.3561 - val_accuracy: 0.8878 - lr: 0.0090
Epoch 3/100
468/468 [==============================] - 22s 47ms/step - loss: 0.2747 - accuracy: 0.9041 - val_loss: 0.4200 - val_accuracy: 0.8547 - lr: 0.0082
Epoch 4/100
468/468 [==============================] - 22s 47ms/step - loss: 0.2491 - accuracy: 0.9137 - val_loss: 0.2026 - val_accuracy: 0.9331 - lr: 0.0074
Epoch 5/100
468/468 [==============================] - 22s 46ms/step - loss: 0.2253 - accuracy: 0.9215 - val_loss: 0.2482 - val_accuracy: 0.9263 - lr: 0.0067
Epoch 6/100
468/468 [==============================] - 21s 44ms/step - loss: 0.2083 - accuracy: 0.9290 - val_loss: 0.1776 - val_accuracy: 0.9384 - lr: 0.0061
Epoch 7/100


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d rahulsinhat/kmnist
!unzip kmnist.zip

 63% 5.00M/7.95M [00:00<00:00, 36.0MB/s]
100% 7.95M/7.95M [00:00<00:00, 54.2MB/s]
Archive:  kmnist.zip
  inflating: X_kannada_MNIST_test.npz  
  inflating: X_kannada_MNIST_train.npz  
  inflating: y_kannada_MNIST_test.npz  
  inflating: y_kannada_MNIST_train.npz  


In [ ]:
import sys
import os
from PyQt5.QtWidgets import QApplication, QMainWindow, QLabel, QLineEdit, QPushButton, QVBoxLayout, QWidget, QFileDialog, QSpinBox
from PyQt5.QtCore import Qt
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

class ImageGeneratorUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Image Generator")
        self.source_folder_path = ""
        self.destination_folder_path = ""
        self.rotation_range = 40
        self.width_shift_range = 0.2
        self.height_shift_range = 0.2
        self.shear_range = 0.2
        self.zoom_range = 0.2
        self.horizontal_flip = True
        self.fill_mode = 'nearest'
        self.num_generated_images = 20

        # Central Widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)

        # Layout
        layout = QVBoxLayout()
        central_widget.setLayout(layout)

        # Source Folder
        source_label = QLabel("Source Folder:")
        layout.addWidget(source_label)
        self.source_folder_entry = QLineEdit()
        layout.addWidget(self.source_folder_entry)
        browse_source_button = QPushButton("Browse")
        browse_source_button.clicked.connect(self.browse_source_folder)
        layout.addWidget(browse_source_button)

        # Destination Folder
        destination_label = QLabel("Destination Folder:")
        layout.addWidget(destination_label)
        self.destination_folder_entry = QLineEdit()
        layout.addWidget(self.destination_folder_entry)
        browse_destination_button = QPushButton("Browse")
        browse_destination_button.clicked.connect(self.browse_destination_folder)
        layout.addWidget(browse_destination_button)

        # Parameters
        parameters_label = QLabel("Parameters:")
        layout.addWidget(parameters_label)
        self.rotation_spinbox = QSpinBox()
        self.rotation_spinbox.setRange(0, 360)
        self.rotation_spinbox.setValue(self.rotation_range)
        self.rotation_spinbox.setSingleStep(10)
        self.rotation_spinbox.setAlignment(Qt.AlignCenter)
        layout.addWidget(self.rotation_spinbox)

        # Generate Button
        generate_button = QPushButton("Generate Images")
        generate_button.clicked.connect(self.generate_images)
        layout.addWidget(generate_button)

        # Number of Generated Images
        num_images_label = QLabel("Number of Images per Original:")
        layout.addWidget(num_images_label)
        self.num_images_spinbox = QSpinBox()
        self.num_images_spinbox.setRange(1, 100)
        self.num_images_spinbox.setValue(self.num_generated_images)
        self.num_images_spinbox.setAlignment(Qt.AlignCenter)
        layout.addWidget(self.num_images_spinbox)

    def browse_source_folder(self):
        self.source_folder_path = QFileDialog.getExistingDirectory(self, "Select Source Folder")
        if self.source_folder_path:
            self.source_folder_entry.setText(self.source_folder_path)

    def browse_destination_folder(self):
        self.destination_folder_path = QFileDialog.getExistingDirectory(self, "Select Destination Folder")
        if self.destination_folder_path:
            self.destination_folder_entry.setText(self.destination_folder_path)

    def generate_images(self):
        # Load parameters
        self.rotation_range = self.rotation_spinbox.value()
        self.num_generated_images = self.num_images_spinbox.value()

        # Define data generator
        datagen = ImageDataGenerator(
            rotation_range=self.rotation_range,
            width_shift_range=self.width_shift_range,
            height_shift_range=self.height_shift_range,
            shear_range=self.shear_range,
            zoom_range=self.zoom_range,
            horizontal_flip=self.horizontal_flip,
            fill_mode=self.fill_mode
        )

        if not (self.source_folder_path and self.destination_folder_path):
            print("Please select source and destination folders.")
            return

        # Generate images
        files = os.listdir(self.source_folder_path)
        for file in files:
            img = load_img(os.path.join(self.source_folder_path, file))
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)

            i = 0
            for batch in datagen.flow(x, batch_size=1,
                                      save_to_dir=self.destination_folder_path, save_prefix='COVID' + file[:-4], save_format='png'):
                i += 1
                if i > self.num_generated_images:
                    break

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ImageGeneratorUI()
    window.show()
    sys.exit(app.exec_())


ModuleNotFoundError: No module named 'PyQt5'

In [ ]:
!pip in

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 kB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 11.0 MB/s eta 0:00:00


In [3]:
from dotenv import load_dotenv
load_dotenv()

import streamlit as st

import google.generativeai as genai
import os


genai.configure(api_key=os.getenv)

# function to lode Gemini Pro midel and get response
model = genai.GenerativeModel('gemini-pro')
chat = model.start_chat(history=[])


def get_gemini_response(question):
    response = chat.send_message(question,stream=True)
    return response

#initilize the streamlit app
st.set_page_config(page_title="Q & A")
st.header("Gemini LLM Application")

# Initilize session state for chat history if it doesnt exist
if 'chat_history' not in st.session_state:
    st.session_state['chat_history'] = []



input = st.text_input("Input:",key="input")
submit = st.button("Ask ⫸")

if submit and input:
    response=get_gemini_response(input)
    # Add user query and response to session chat history
    st.session_state['chat_history'].append(("You",input))
    st.subheader("The Response is")
    for chunk in response:
        st.write(chunk.text)
        st.session_state['chat_history'].append(("Bot",chunk.text))

st.subheader("The Chat History is")

for role,text in st.session_state['chat_history']:
    st.write(f"{role}:{text}")

ModuleNotFoundError: No module named 'dotenv'

In [2]:
!pip install dotenv

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
